In [ ]:
%pip install langchain langchain-openai openai python-dotenv

## Import Libraries 

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

# Load environment variables from .env file
load_dotenv()


True

## BASIC CHAT MODEL INVOCATION

In [3]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [4]:
messages = [
    SystemMessage(content="You are a concise backend engineering expert."),
    HumanMessage(content="In one sentence, what is a B-tree index?"),
]

In [7]:
response = llm.invoke(messages)
print("=== Direct Invocation ===")
print(response.content)
print(f"Model used: {response.response_metadata['model_name']}")
print(f"Tokens used: {response.response_metadata['token_usage']}")
print()

=== Direct Invocation ===
A B-tree index is a balanced multi-way tree that stores keys in sorted order across internal nodes and leaves to enable fast, logarithmic search, insert, delete, and range queries.
Model used: gpt-5-nano-2025-08-07
Tokens used: {'completion_tokens': 366, 'prompt_tokens': 29, 'total_tokens': 395, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}



## PROMPT TEMPLATES

In [9]:
template_explicit = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise, brutal, and technical."),
    ("human", "explain {concept} in 2 sentences."),
])

template_simple = ChatPromptTemplate.from_template(
    "Explain {concept} to a {audience} in 2 sentences."
)

print("=== Prompt Template ===")
formatted = template_explicit.invoke({"domain": "database", "concept": "WAL logging"})
print(formatted.messages)
print()
# formatted = template_simple.invoke({"concept": "WAL logging", "audience": "backend engineer"})
# print(formatted.messages)   
# print()

=== Prompt Template ===
[SystemMessage(content='You are an expert in database. Be concise, brutal, and technical.', additional_kwargs={}, response_metadata={}), HumanMessage(content='explain WAL logging in 2 sentences.', additional_kwargs={}, response_metadata={})]



## STROUTPUTPARSER — most common 

In [10]:
chain_str = template_explicit | llm | StrOutputParser()

result = chain_str.invoke({
    "domain": "distributed systems", 
    "concept": "CAP theorem"
    })

print("=== Chain with String Output Parser ===")
print(result)
print(type(result))
print()

=== Chain with String Output Parser ===
The CAP theorem states that in a distributed system subject to network partitions, you cannot simultaneously guarantee Consistency, Availability, and Partition Tolerance; when a partition occurs you must sacrifice either Consistency or Availability. In practice, systems pick CP (strong consistency with reduced availability during partitions) or AP (always available with weaker or eventual consistency), often with tunable consistency levels to balance the trade-off.
<class 'langchain_core.messages.base.TextAccessor'>



## JSONOUTPUTPARSER

In [13]:
json_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always respond in valid JSON only. No markdown, no explanation."),
    ("human", "Give me info about {language}. Return JSON with keys: name, year_created, creator, main_use_case."),
])

chain_json = json_template | llm | JsonOutputParser()

result_json = chain_json.invoke({"language": "Python"})
print("=== Chain with JSON Output Parser ===")
print(result_json)
print(type(result_json))
print(f"Creator: {result_json['creator']}")
print()

=== Chain with JSON Output Parser ===
{'name': 'Python', 'year_created': 1991, 'creator': 'Guido van Rossum', 'main_use_case': 'General-purpose programming language for scripting, automation, software development, data analysis, web development, and education.'}
<class 'dict'>
Creator: Guido van Rossum



## PYDANTICOUTPUTPARSER — most powerful

In [14]:
class TechSummary(BaseModel):
    name: str = Field(description="Name of the technology")
    category: str = Field(description="Category of the technology, e.g., database, programming language")
    pros: str = Field(description="One major advantage of the technology")
    cons: str = Field(description="One major disadvantage of the technology")
    use_when: str = Field(description="One scenario where this technology is particularly useful")

parser_pydantic = PydanticOutputParser(pydantic_object=TechSummary)

pydantic_template = ChatPromptTemplate.from_messages([
    ("system", "you are a tech expert. {format_instructions}"),
    ("human", "Give me a summary of {technology} for a software engineer.")
])

chain_pydantic = pydantic_template | llm | parser_pydantic

result_pydantic = chain_pydantic.invoke({
    "technology": "redis",
    "format_instructions": parser_pydantic.get_format_instructions()
    })

print("=== Chain with Pydantic Output Parser ===")
print(result_pydantic)
print(type(result_pydantic))
print(f"Name: {result_pydantic.name}")
print(f"Category: {result_pydantic.category}")
print(f"Pros: {result_pydantic.pros}")
print(f"Cons: {result_pydantic.cons}")
print(f"Use When: {result_pydantic.use_when}")
print()

=== Chain with Pydantic Output Parser ===
name='Redis' category='In-memory data store / cache' pros='Extremely fast in-memory data store with rich data structures (strings, hashes, lists, sets, sorted sets, streams) and atomic operations.' cons='Primarily in-memory; durability and memory size must be managed (persistence is optional but not guaranteed without configuration); not a full relational database.' use_when='You need ultra-fast reads/writes and real-time features (caching, sessions, leaderboards, queues, pub/sub) with simple data models and atomic operations.'
<class '__main__.TechSummary'>
Name: Redis
Category: In-memory data store / cache
Pros: Extremely fast in-memory data store with rich data structures (strings, hashes, lists, sets, sorted sets, streams) and atomic operations.
Cons: Primarily in-memory; durability and memory size must be managed (persistence is optional but not guaranteed without configuration); not a full relational database.
Use When: You need ultra-fas